# Multivariate Statistics

## Learning Objectives
1. Understand the multivariate Gaussian distribution: mean vector, covariance matrix, and confidence ellipses
2. Implement PCA from scratch using eigendecomposition and compare to sklearn PCA
3. Apply Mahalanobis distance for anomaly detection and compare to Euclidean distance
4. Diagnose multicollinearity using VIF and demonstrate its effect on regression coefficients

In [ ]:
# Cell 2: Imports and setup
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification
from sklearn.linear_model import LinearRegression
from sklearn.covariance import MinCovDet

np.random.seed(42)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Libraries loaded.')

## Level 1: Multivariate Gaussian — Sampling, Covariance, and Confidence Ellipses

Visualize the structure of a 2D Gaussian with different covariance matrices and draw confidence ellipses at the 1-sigma and 2-sigma levels.

In [ ]:
# Cell 4: Multivariate Gaussian basics
# Sample from N(mu, Sigma) and visualize the confidence ellipse
# Confidence ellipse: set of x where (x-mu)^T Sigma^-1 (x-mu) = chi2(p, alpha)

def draw_confidence_ellipse(ax, mu: np.ndarray, Sigma: np.ndarray,
                             n_std: float = 2.0, **kwargs):
    """Draw a confidence ellipse for a 2D Gaussian.
    
    The ellipse axes come from the eigendecomposition of Sigma.
    n_std=1 encloses ~39%, n_std=2 encloses ~86% of probability mass in 2D.
    """
    eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
    # Rotation angle from the dominant eigenvector
    angle = np.degrees(np.arctan2(eigenvectors[1, 1], eigenvectors[0, 1]))
    # Semi-axes: sqrt of eigenvalues scaled by n_std
    width = 2 * n_std * np.sqrt(eigenvalues[1])
    height = 2 * n_std * np.sqrt(eigenvalues[0])
    ellipse = Ellipse(mu, width=width, height=height, angle=angle, **kwargs)
    ax.add_patch(ellipse)
    return ellipse


# Three different covariance structures
configs = [
    {'name': 'No correlation (I)', 'Sigma': np.array([[1.0, 0.0], [0.0, 1.0]])},
    {'name': 'Positive corr (0.8)', 'Sigma': np.array([[1.5, 1.1], [1.1, 1.0]])},
    {'name': 'Negative corr (-0.7)', 'Sigma': np.array([[2.0, -1.2], [-1.2, 1.0]])},
]
mu = np.array([0.0, 0.0])
rng = np.random.default_rng(42)
N_SAMP = 500

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, cfg in zip(axes, configs):
    Sigma = cfg['Sigma']
    # Validate positive semi-definite
    eigs = np.linalg.eigvalsh(Sigma)
    assert np.all(eigs >= -1e-10), f'Sigma is not PSD: eigenvalues={eigs}'
    # Sample from N(mu, Sigma)
    samples = rng.multivariate_normal(mu, Sigma, N_SAMP)
    # Compute Mahalanobis distances
    Sigma_inv = np.linalg.inv(Sigma)
    diff = samples - mu
    mahal_sq = np.sum(diff @ Sigma_inv * diff, axis=1)
    # Plot samples colored by Mahalanobis distance
    sc = ax.scatter(samples[:, 0], samples[:, 1],
                    c=np.sqrt(mahal_sq), cmap='viridis', s=15, alpha=0.6)
    plt.colorbar(sc, ax=ax, label='Mahalanobis dist')
    # Confidence ellipses
    for nstd, alpha_val, ls in [(1, 0.3, 'solid'), (2, 0.15, 'dashed')]:
        draw_confidence_ellipse(ax, mu, Sigma, n_std=nstd,
                                fill=True, alpha=alpha_val, color='red',
                                linestyle=ls, edgecolor='red', lw=2)
    ax.set_title(cfg['name'])
    ax.set_xlabel('X1')
    ax.set_ylabel('X2')
    # Print actual sample correlation
    emp_corr = np.corrcoef(samples[:, 0], samples[:, 1])[0, 1]
    ax.text(0.05, 0.95, f'Empirical corr: {emp_corr:.2f}',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Multivariate Gaussian: Samples and Confidence Ellipses (1-sigma, 2-sigma)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('mvn_ellipses.png', dpi=80, bbox_inches='tight')
plt.show()

print('Samples with Mahal. dist > 2.45 are outside the 2-sigma ellipse (chi2(2, 0.95) = 5.99, sqrt = 2.45)')
print('Level 1 complete.')

## Level 2: PCA from Scratch vs sklearn

Implement PCA via eigendecomposition of the covariance matrix, compare results to sklearn, and plot explained variance.

In [ ]:
# Cell 6: PCA from scratch and comparison to sklearn PCA
# PCA steps: center → covariance → eigendecomposition → project

def pca_from_scratch(X: np.ndarray, n_components: int) -> dict:
    """PCA via covariance matrix eigendecomposition.
    
    Args:
        X: Data matrix of shape (n_samples, n_features)
        n_components: Number of principal components to keep
    
    Returns:
        Dict with 'projection', 'explained_var_ratio', 'loadings'
    """
    n, p = X.shape
    # Step 1: Center the data (subtract column means)
    X_centered = X - X.mean(axis=0)
    # Step 2: Compute sample covariance matrix (divide by n-1)
    Sigma = (X_centered.T @ X_centered) / (n - 1)
    # Step 3: Eigendecompose the covariance matrix
    # np.linalg.eigh returns eigenvalues in ascending order for symmetric matrices
    eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
    # Step 4: Sort by descending eigenvalue (most variance first)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    # Step 5: Compute explained variance ratio
    explained_var_ratio = eigenvalues / eigenvalues.sum()
    # Step 6: Project data onto top-k principal components
    components = eigenvectors[:, :n_components]  # shape: (p, k)
    X_proj = X_centered @ components             # shape: (n, k)
    return {
        'projection': X_proj,
        'explained_var_ratio': explained_var_ratio,
        'cumulative_var': np.cumsum(explained_var_ratio),
        'eigenvalues': eigenvalues,
        'loadings': components,
    }


# Generate 5D correlated data (known low-dim structure)
rng = np.random.default_rng(7)
n_samples = 300
# True 2D structure: two hidden factors
latent = rng.normal(0, 1, (n_samples, 2))
# Mix into 5D space with some correlation
mixing = np.array([[1, 0.5], [0.8, 0.2], [-0.6, 0.9], [0.3, -0.7], [0.1, 1.1]])
X_raw = latent @ mixing.T + rng.normal(0, 0.3, (n_samples, 5))

# Standardize before PCA (important when features have different scales)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# PCA from scratch
pca_scratch = pca_from_scratch(X_scaled, n_components=2)

# sklearn PCA for comparison
pca_sk = PCA(n_components=5)
pca_sk.fit(X_scaled)
X_sk_proj = pca_sk.transform(X_scaled)[:, :2]

print('PCA from scratch vs sklearn (explained variance ratio per component):')
print(f'  {'PC':>4}  {'Scratch':>10}  {'sklearn':>10}  {'Cumulative (Scratch)':>22}')
print('-' * 52)
for i in range(5):
    print(f'  {i+1:>4}  {pca_scratch["explained_var_ratio"][i]:>10.4f}  '
          f'{pca_sk.explained_variance_ratio_[i]:>10.4f}  '
          f'{pca_scratch["cumulative_var"][i]:>22.4f}')

# Find number of components for 95% explained variance
n_95 = np.argmax(pca_scratch['cumulative_var'] >= 0.95) + 1
print(f'\nComponents needed for 95% explained variance: {n_95}')

# Verify projections agree (up to sign flip)
corr_pc1 = abs(np.corrcoef(pca_scratch['projection'][:, 0], X_sk_proj[:, 0])[0, 1])
corr_pc2 = abs(np.corrcoef(pca_scratch['projection'][:, 1], X_sk_proj[:, 1])[0, 1])
print(f'Correlation between scratch and sklearn projections: PC1={corr_pc1:.6f}, PC2={corr_pc2:.6f}')

# Plot scree and explained variance
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(range(1, 6), pca_scratch['explained_var_ratio'], color='navy', alpha=0.8, label='Individual')
ax2 = axes[0].twinx()
ax2.plot(range(1, 6), pca_scratch['cumulative_var'], 'r-o', lw=2, label='Cumulative')
ax2.axhline(0.95, color='red', ls='--', lw=1, label='95% threshold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
ax2.set_ylabel('Cumulative Explained Variance')
axes[0].set_title('Scree Plot')
axes[0].legend(loc='upper right')
ax2.legend(loc='center right')

# 2D projection scatter
axes[1].scatter(pca_scratch['projection'][:, 0], pca_scratch['projection'][:, 1],
                color='navy', alpha=0.5, s=15)
axes[1].set_xlabel(f'PC1 ({pca_scratch["explained_var_ratio"][0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca_scratch["explained_var_ratio"][1]*100:.1f}%)')
axes[1].set_title('Data Projected onto Top 2 PCs')

plt.tight_layout()
plt.savefig('pca_results.png', dpi=80, bbox_inches='tight')
plt.show()
print('Level 2 complete.')

## Real-World Example 1: Anomaly Detection via Mahalanobis Distance

Fit a multivariate Gaussian to normal data, compute Mahalanobis distances for all points, and flag anomalies using the chi-squared threshold.

In [ ]:
# Cell 8: Anomaly detection via Mahalanobis distance
# Strategy: fit N(mu, Sigma) to normal data; flag points where d^2 > chi2(p, 0.975)

rng = np.random.default_rng(55)

# Generate normal operating data (bivariate Gaussian)
N_NORMAL = 400
mu_normal = np.array([10.0, 5.0])
Sigma_normal = np.array([[4.0, 2.4], [2.4, 2.0]])  # Correlated features
X_normal = rng.multivariate_normal(mu_normal, Sigma_normal, N_NORMAL)

# Inject anomalies (points from a shifted distribution)
N_ANOM = 20
anomalies = rng.multivariate_normal([15.0, 8.0], np.eye(2) * 0.5, N_ANOM)
X_all = np.vstack([X_normal, anomalies])
true_labels = np.array([0] * N_NORMAL + [1] * N_ANOM)  # 0=normal, 1=anomaly

# Fit parameters on normal data only (not including anomalies)
mu_hat = X_normal.mean(axis=0)
Sigma_hat = np.cov(X_normal.T)
Sigma_hat_inv = np.linalg.inv(Sigma_hat)

# Compute Mahalanobis distance for all points (including anomalies)
def mahalanobis_distance(X: np.ndarray, mu: np.ndarray, Sigma_inv: np.ndarray) -> np.ndarray:
    """Compute Mahalanobis distance for each row in X.
    
    d^2 = (x - mu)^T Sigma^-1 (x - mu)
    Returns array of distances, not squared distances.
    """
    diff = X - mu
    # Vectorized computation: diag(diff @ Sigma_inv @ diff.T)
    mahal_sq = np.sum(diff @ Sigma_inv * diff, axis=1)
    return np.sqrt(np.maximum(mahal_sq, 0))  # clip for numerical stability


mahal_dists = mahalanobis_distance(X_all, mu_hat, Sigma_hat_inv)
eucl_dists = np.linalg.norm(X_all - mu_hat, axis=1)

# Threshold: chi2(p=2, alpha=0.975) => sqrt gives Mahalanobis threshold
p_features = 2
alpha = 0.975
chi2_threshold = np.sqrt(stats.chi2.ppf(alpha, df=p_features))
print(f'Chi-squared threshold at {alpha:.0%} level (df={p_features}): {chi2_threshold:.3f}')

# Predictions
mahal_preds = (mahal_dists > chi2_threshold).astype(int)
eucl_preds = (eucl_dists > np.percentile(eucl_dists, 97.5)).astype(int)

# Evaluate precision and recall for anomaly detection
def eval_detector(preds, true_labels):
    tp = np.sum((preds == 1) & (true_labels == 1))
    fp = np.sum((preds == 1) & (true_labels == 0))
    fn = np.sum((preds == 0) & (true_labels == 1))
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    return precision, recall, tp, fp, fn

m_prec, m_rec, m_tp, m_fp, m_fn = eval_detector(mahal_preds, true_labels)
e_prec, e_rec, e_tp, e_fp, e_fn = eval_detector(eucl_preds, true_labels)

print(f'\n{'Method':<20}  {'Precision':>10}  {'Recall':>8}  {'TP':>4}  {'FP':>4}  {'FN':>4}')
print('-' * 56)
print(f'{'Mahalanobis':<20}  {m_prec:>10.3f}  {m_rec:>8.3f}  {m_tp:>4}  {m_fp:>4}  {m_fn:>4}')
print(f'{'Euclidean (97.5%)':<20}  {e_prec:>10.3f}  {e_rec:>8.3f}  {e_tp:>4}  {e_fp:>4}  {e_fn:>4}')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (preds, method) in zip(axes, [
    (mahal_preds, f'Mahalanobis (thresh={chi2_threshold:.2f})'),
    (eucl_preds, 'Euclidean (97.5th percentile)')
]):
    # True normals correctly classified
    ax.scatter(X_all[(true_labels==0)&(preds==0), 0],
               X_all[(true_labels==0)&(preds==0), 1],
               color='steelblue', s=15, alpha=0.5, label='Normal (correct)')
    # True anomalies detected
    ax.scatter(X_all[(true_labels==1)&(preds==1), 0],
               X_all[(true_labels==1)&(preds==1), 1],
               color='red', s=60, marker='x', zorder=5, label='Anomaly (detected)')
    # Missed anomalies
    ax.scatter(X_all[(true_labels==1)&(preds==0), 0],
               X_all[(true_labels==1)&(preds==0), 1],
               color='orange', s=60, marker='x', zorder=5, label='Anomaly (missed)')
    # False positives
    ax.scatter(X_all[(true_labels==0)&(preds==1), 0],
               X_all[(true_labels==0)&(preds==1), 1],
               color='purple', s=30, alpha=0.7, label='False positive')
    ax.set_title(f'{method}\nPrec={eval_detector(preds, true_labels)[0]:.2f}, Rec={eval_detector(preds, true_labels)[1]:.2f}')
    ax.legend(fontsize=8)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('Anomaly Detection: Mahalanobis vs Euclidean Distance', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('anomaly_detection.png', dpi=80, bbox_inches='tight')
plt.show()

## Real-World Example 2: Multicollinearity Diagnosis with VIF

Show how highly correlated features inflate regression coefficient variance (VIF effect), and how Ridge regression or feature removal remedies it.

In [ ]:
# Cell 10: VIF diagnosis and multicollinearity effects on regression
# VIF_j = 1/(1 - R^2_j) where R^2_j = regress feature j on all other features
# VIF > 10 => severe multicollinearity

from sklearn.linear_model import Ridge


def compute_vif(X: np.ndarray) -> np.ndarray:
    """Compute Variance Inflation Factor for each feature in X.
    
    VIF_j = 1 / (1 - R2_j) where R2_j is R-squared from regressing
    feature j on all other features. VIF > 10 indicates serious multicollinearity.
    """
    n_features = X.shape[1]
    vif = np.zeros(n_features)
    for j in range(n_features):
        # Regress feature j on all others
        X_other = np.delete(X, j, axis=1)
        reg = LinearRegression().fit(X_other, X[:, j])
        r2 = reg.score(X_other, X[:, j])
        # Avoid division by zero for perfectly collinear features
        vif[j] = 1.0 / (1 - r2) if r2 < 1 - 1e-10 else 1e6
    return vif


rng = np.random.default_rng(42)
N = 200

# Generate features with varying degrees of correlation
z = rng.normal(0, 1, N)  # Hidden shared factor
x1 = z + rng.normal(0, 0.3, N)       # Highly correlated with x2
x2 = z + rng.normal(0, 0.3, N)       # Highly correlated with x1
x3 = rng.normal(0, 1, N)              # Independent feature
x4 = 0.9 * x1 + rng.normal(0, 0.1, N)  # Almost perfectly collinear with x1

# True regression: y depends on x1 and x3 only
true_coefs = np.array([1.0, 0.0, 2.0, 0.0])
y = true_coefs[0]*x1 + true_coefs[1]*x2 + true_coefs[2]*x3 + true_coefs[3]*x4 + rng.normal(0, 0.5, N)

X = np.column_stack([x1, x2, x3, x4])
feature_names = ['x1', 'x2', 'x3', 'x4']

# VIF for each feature
vif_values = compute_vif(X)
print('Variance Inflation Factors:')
for name, vif in zip(feature_names, vif_values):
    flag = ' *** SEVERE (VIF > 10)' if vif > 10 else (' * elevated' if vif > 5 else '')
    print(f'  {name}: VIF = {vif:.2f}{flag}')

# Compare OLS vs Ridge coefficients
ols = LinearRegression().fit(X, y)
ridge_strong = Ridge(alpha=10.0).fit(X, y)

print(f'\n{'Feature':<8}  {'True':>8}  {'OLS':>10}  {'Ridge(10)':>12}')
print('-' * 46)
for i, name in enumerate(feature_names):
    print(f'{name:<8}  {true_coefs[i]:>8.3f}  {ols.coef_[i]:>10.3f}  {ridge_strong.coef_[i]:>12.3f}')

# Bootstrap to show OLS coefficient variance with multicollinearity
n_boot = 300
ols_boot_coefs = []
for _ in range(n_boot):
    idx = rng.choice(N, N, replace=True)
    reg = LinearRegression().fit(X[idx], y[idx])
    ols_boot_coefs.append(reg.coef_)
ols_boot_coefs = np.array(ols_boot_coefs)

print(f'\nBootstrap std dev of OLS coefficients (measure of instability):')
for i, name in enumerate(feature_names):
    print(f'  {name}: std = {ols_boot_coefs[:, i].std():.4f}')
print('High std = coefficient is unstable due to multicollinearity.')

## Real-World Example 3: PCA for Visualization + Comparison Summary

Use PCA to reduce high-dimensional data for 2D visualization, compare Euclidean vs Mahalanobis distance for outlier detection across multiple scenarios, and summarize trade-offs.

In [ ]:
# Cell 12: PCA visualization + comprehensive comparison of methods

# ---- Part A: PCA visualization on classification dataset ----
rng = np.random.default_rng(77)

# Generate a 10-dimensional dataset with 3 classes
X_class, y_class = make_classification(
    n_samples=300, n_features=10, n_informative=4, n_redundant=3,
    n_classes=3, n_clusters_per_class=1, random_state=42
)

# Standardize before PCA
X_std = StandardScaler().fit_transform(X_class)

# PCA to 2D
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_std)

print(f'10D to 2D PCA: explained variance = {pca_2d.explained_variance_ratio_.sum()*100:.1f}%')

# ---- Part B: Comprehensive comparison — Euclidean vs Mahalanobis ----
# Scenario 1: Spherical data — both methods agree
# Scenario 2: Correlated data — Mahalanobis correctly handles correlation
# Scenario 3: Different scales — Mahalanobis is scale-invariant, Euclidean is not

scenarios = [
    {
        'name': 'Spherical (I)',
        'Sigma': np.eye(2),
        'mu': np.zeros(2),
    },
    {
        'name': 'Correlated (0.9)',
        'Sigma': np.array([[1.0, 0.9], [0.9, 1.0]]),
        'mu': np.zeros(2),
    },
    {
        'name': 'Different scales',
        'Sigma': np.array([[100.0, 0.0], [0.0, 0.01]]),
        'mu': np.zeros(2),
    },
]

# Test point: clearly anomalous in Mahalanobis sense
test_points = [
    np.array([2.5, 2.5]),
    np.array([2.5, -0.5]),  # Near-diagonal point in correlated case
    np.array([5.0, 0.05]),  # Large in x1 but within-scale for that feature
]

print(f'\n{'Scenario':<22}  {'Eucl. dist':>12}  {'Mahal. dist':>13}  {'Eucl>2.0':>8}  {'Mahal>2.45':>10}')
print('-' * 72)
for sc, tp in zip(scenarios, test_points):
    Sigma = sc['Sigma']
    mu = sc['mu']
    Sigma_inv = np.linalg.inv(Sigma)
    eucl = np.linalg.norm(tp - mu)
    diff = tp - mu
    mahal = np.sqrt(diff @ Sigma_inv @ diff)
    eucl_flag = 'OUTLIER' if eucl > 2.0 else 'normal'
    mahal_flag = 'OUTLIER' if mahal > 2.45 else 'normal'
    print(f'{sc["name"]:<22}  {eucl:>12.3f}  {mahal:>13.3f}  {eucl_flag:>8}  {mahal_flag:>10}')

# Summary table: trade-offs between methods
print('\n=== Method Summary ===')
print(f'{'Method':<25}  {'Corr-aware':>10}  {'Scale-inv':>10}  {'Needs Sigma':>12}  {'High-dim':>9}')
print('-' * 72)
rows = [
    ('Euclidean', 'No', 'No', 'No', 'Yes'),
    ('Std. Euclidean', 'No', 'Yes', 'Var only', 'Yes'),
    ('Mahalanobis', 'Yes', 'Yes', 'Full Sigma', 'Problematic'),
    ('PCA + Euclidean', 'Yes', 'Yes (post)', 'PCA fit', 'Reduced'),
]
for r in rows:
    print(f'{r[0]:<25}  {r[1]:>10}  {r[2]:>10}  {r[3]:>12}  {r[4]:>9}')

# Final visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA visualization
colors_3 = ['navy', 'firebrick', 'green']
for c in range(3):
    mask = y_class == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    color=colors_3[c], s=20, alpha=0.6, label=f'Class {c}')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('PCA: 10D → 2D Visualization')
axes[0].legend()

# VIF visualization from Part 2
# (Re-use vif_values from Cell 10)
axes[1].barh(feature_names, vif_values, color=['red' if v > 10 else 'steelblue' for v in vif_values])
axes[1].axvline(10, color='red', ls='--', lw=2, label='VIF=10 threshold')
axes[1].axvline(5, color='orange', ls='--', lw=1.5, label='VIF=5 caution')
axes[1].set_xlabel('VIF')
axes[1].set_title('Variance Inflation Factors by Feature')
axes[1].legend()

plt.suptitle('Multivariate Statistics: PCA + Multicollinearity', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('multivariate_comparison.png', dpi=80, bbox_inches='tight')
plt.show()

print('\nKey Takeaways:')
print('- Mahalanobis distance corrects for correlation and scale; use for anomaly detection')
print('- VIF > 10 signals severe multicollinearity; remove or regularize affected features')
print('- PCA requires standardization unless all features share the same scale')
print('- Never fit PCA on full data before splitting; fit on train, transform test')